## Demo 5: ProjectStatus

One Excel file, filled in by hand by eight project managers, and a table with constraints that says
what a project status is allowed to look like.

The first four demos moved data that was *right*. Sample data from an archive, geometry from a GPX
file, an application's own tables - all of it already fit the target. This one does not, and that is
the entire scenario: **what do you do when some of the rows are wrong?**

### What data do we have?

In [ ]:
import os

data_path = r"..\data\projectstatus"

for entry in sorted(os.listdir(data_path)):
    print(f"{entry:30} {os.path.getsize(os.path.join(data_path, entry)):>7} bytes")

The file is a form. Row 1 is a note to the person filling it in, row 2 is empty, and the header is on
row 3 - so the reader skips two rows, exactly as the Timesheets importer does.

In [ ]:
# os.startfile(os.path.join(data_path, "ProjectStatus.xlsx"))

In [ ]:
import pandas as pd

data = pd.read_excel(
    os.path.join(data_path, "ProjectStatus.xlsx"),
    sheet_name="ProjectStatus",
    skiprows=2
)

data

Twelve rows, and three of them are blank - the managers left a gap between the sections. The sibling
passes `-DataOnly` to `Import-Excel` and never sees them; here they arrive as rows of `NaN` and
`dropna(how="all")` is what removes them.

What is left is nine rows, and one of those is `NEW PROJECTS:` - a heading somebody typed into the
`Title` column.

In [ ]:
data = data.dropna(how="all")

print(len(data), "rows")
data["Title"].tolist()

### The target table

This is where the rules live. Not in the spreadsheet, and not in the import script - in the table:

* `Title` is the primary key
* `Priority` may only be `Low`, `Medium` or `High`
* `Color` may only be `Green`, `Yellow` or `Red`
* `ProgressPercent` must be between 0 and 100
* `Status` is a `VARCHAR(50)`, and `MilestoneDate` is a `DATETIME2`

Every one of those is a chance for a hand-filled form to be rejected.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("../lib").resolve()))

from connect_sql_instance import connect_sql_instance
from invoke_sql_query import invoke_sql_query
from write_sql_table import write_sql_table

sql_connection = connect_sql_instance(
    instance="127.0.0.1",
    database="ProjectStatus",
    username="ProjectStatus",
    password="Passw0rd!"
)

In [ ]:
create_query = """
CREATE TABLE dbo.ProjectStatus (
  Title            VARCHAR(50),
  Priority         VARCHAR(10),
  Manager          VARCHAR(50),
  Status           VARCHAR(50),
  Color            VARCHAR(10),
  ProgressPercent  INT,
  Milestone        VARCHAR(100),
  MilestoneDate    DATETIME2,
  CONSTRAINT ProjectStatus_PK
  PRIMARY KEY (Title),
  CONSTRAINT ProjectStatus_Priority
  CHECK (Priority IN ('Low', 'Medium', 'High')),
  CONSTRAINT ProjectStatus_Color
  CHECK (Color IN ('Green', 'Yellow', 'Red')),
  CONSTRAINT ProjectStatus_ProgressPercent
  CHECK (ProgressPercent >= 0 AND ProgressPercent <= 100)
)
"""

invoke_sql_query(connection=sql_connection, query=create_query)

### The obvious way, which does not work

`write_sql_table` is what the first four demos used, and it is the right tool whenever the data fits.
Here it does not.

In [ ]:
write_sql_table(
    connection=sql_connection,
    table="dbo.ProjectStatus",
    data=data
)

In [ ]:
invoke_sql_query(connection=sql_connection, query="SELECT * FROM dbo.ProjectStatus")

Nothing was imported - not even the rows that were fine.

That is what a bulk load *is*: one statement, one answer. It reports the first thing it could not do
and stops, so the good rows are lost along with the bad ones, and the message names one problem when
there are several.

Look at what the message actually is, though. A right truncation with two byte counts in it, naming
neither the column it happened in nor the row it came from - `fast_executemany` sized its buffer from
the target column, a value did not fit, and that is genuinely all it knows.

The sibling fails earlier and differently. `Write-SqlTable` never reaches the database: filling its
`DataTable` throws first, and it gets to say *"The string 'Late july 2026' was not recognized as a
valid DateTime"* - a better message, about a different one of the four mistakes. Neither version
tells you which row to go and look at.

### One row at a time

If the rows have to be judged individually, they have to be sent individually. A small function keeps
the loop readable - the sibling defines exactly this one, in exactly this place.

The `None if pd.isna(value) else value` is the one line that has no counterpart in the sibling.
pandas writes a missing cell as `NaN`, which is a **float**, and binding a float into a `VARCHAR`
column is not what anybody meant. `Import-Excel` hands PowerShell a `$null` and the question never
comes up.

In [ ]:
def import_projectstatus_row(
    connection,
    row,
    enable_exception=False
):
    query = """
    INSERT INTO dbo.ProjectStatus (Title, Priority, Manager, Status, Color, ProgressPercent, Milestone, MilestoneDate)
    VALUES (@Title, @Priority, @Manager, @Status, @Color, @ProgressPercent, @Milestone, @MilestoneDate)
    """

    # A missing cell is NaN, a float, and the driver would try to bind it as one
    parameter_values = {
        column: None if pd.isna(value) else value
        for column, value in row.items()
    }

    return invoke_sql_query(
        connection=connection,
        query=query,
        parameter_values=parameter_values,
        enable_exception=enable_exception
    )

Now the loop. `NEW PROJECTS:` is skipped, because it is a heading rather than a project - the same
guard the sibling has.

In [ ]:
invoke_sql_query(connection=sql_connection, query="TRUNCATE TABLE dbo.ProjectStatus")

for _, row in data.iterrows():
    if str(row["Title"]).endswith(":"):
        continue

    import_projectstatus_row(connection=sql_connection, row=row)

invoke_sql_query(connection=sql_connection, query="SELECT Title, Color, ProgressPercent, MilestoneDate FROM dbo.ProjectStatus")

Four rows in, four rows rejected - and the good rows are no longer held hostage by the bad ones.

But look at what the loop actually told us. `[ERROR]` lines scrolled past with the database's
complaint in them, and the table shows what arrived, and **neither of those says which row failed**.
Matching four error messages to eight input rows by eye is not a plan.

That is what `enable_exception` is for. Every function in `lib/` takes it, and it is the only
error-handling switch the library has: without it a failure prints and returns `None`, with it the
failure is raised and the caller decides.

In [ ]:
invoke_sql_query(connection=sql_connection, query="TRUNCATE TABLE dbo.ProjectStatus")

for _, row in data.iterrows():
    if str(row["Title"]).endswith(":"):
        continue

    try:
        import_projectstatus_row(connection=sql_connection, row=row, enable_exception=True)
        print(f"OK      {row['Title']}")
    except Exception as error:
        print(f"FAILED  {row['Title']}: {str(error)[:120]}")

Now every row is accounted for, and the four failures are four different mistakes:

* a date column with `Late july 2026` typed into it
* a `Status` longer than the 50 characters the column allows
* a colour that is not one of the three the constraint permits
* the word `unknown` where a percentage belongs

None of these is a bug in the import. They are a form filled in by people, which is what the demo is
about.

### Handing the failures back

Printing them is not enough. Whoever filled the form in needs to see their own rows, with the reason
attached - so collect them instead of just reporting them, and write them to an Excel file of their
own.

In [ ]:
failures_path = os.path.join(data_path, "ProjectStatus_Failures.xlsx")

invoke_sql_query(connection=sql_connection, query="TRUNCATE TABLE dbo.ProjectStatus")

failed_rows = []

for _, row in data.iterrows():
    if str(row["Title"]).endswith(":"):
        continue

    try:
        import_projectstatus_row(connection=sql_connection, row=row, enable_exception=True)
    except Exception as error:
        # The row as it came out of the file, plus why the database would not take it
        failed_rows.append({**row.to_dict(), "ImportError": str(error)})

failures = pd.DataFrame(failed_rows)

failures.to_excel(failures_path, sheet_name="ImportFailures", index=False)

print(f"{len(failures)} rows written to {os.path.basename(failures_path)}")
failures[["Title", "Color", "ProgressPercent", "MilestoneDate"]]

In [ ]:
# os.startfile(failures_path)

`to_excel` writes the frame and nothing else. The sibling asks `Export-Excel` for `-BoldTopRow` and
`-AutoSize` in the same call; pandas has no equivalent, and styling it would mean opening the file
again with openpyxl, exactly as the Timesheets report does.

### Fixing what can be fixed

Some failures are guesses a program is entitled to make. `DarkRed` is not a permitted colour, but
nobody typing it meant *green*.

The error message is how the loop knows which failure it is looking at - the constraint name comes
straight back from SQL Server, so `ProjectStatus_Color` in the text is the signal to try again.

In [ ]:
invoke_sql_query(connection=sql_connection, query="TRUNCATE TABLE dbo.ProjectStatus")

failed_rows = []

for _, row in data.iterrows():
    if str(row["Title"]).endswith(":"):
        continue

    try:
        import_projectstatus_row(connection=sql_connection, row=row, enable_exception=True)
        continue
    except Exception as error:
        message = str(error)

    if "ProjectStatus_Color" in message:
        print(f"RETRY   {row['Title']}: colour {row['Color']} is not allowed, trying Red")
        row = row.copy()
        row["Color"] = "Red"
        try:
            import_projectstatus_row(connection=sql_connection, row=row, enable_exception=True)
            continue
        except Exception as error:
            message = str(error)

    failed_rows.append({**row.to_dict(), "ImportError": message})

failures = pd.DataFrame(failed_rows)
failures.to_excel(failures_path, sheet_name="ImportFailures", index=False)

invoke_sql_query(connection=sql_connection, query="SELECT Title, Color, ProgressPercent, MilestoneDate FROM dbo.ProjectStatus")

Five rows in, three handed back.

The three that are left are the ones a program should not guess at. `Late july 2026` might be the
15th or the 31st; `unknown` is not a number anybody can invent; and a status somebody wrote 78
characters of cannot be shortened without deciding which half matters. Those go back to the person
who typed them.

Where the line sits is a judgement, not a rule. The same loop could just as reasonably clamp
`ProgressPercent` into 0..100, or write `NULL` where a value is unusable and let the column allow it,
or hand the free text to a model and ask what date `Late july 2026` means. The point is that the
decision is visible, in a loop you can read, rather than buried in whatever the bulk load happened to
do.

### Key takeaways

* A bulk load is one statement with one answer. If any row can be wrong, it will take the good rows
  down with it.
* Row by row costs speed and buys you the ability to say *which* row.
* `enable_exception` is the switch that makes that possible: without it a failure is a printed line,
  with it a failure is something the caller can catch.
* The database is where the rules belong. Every failure here came back naming its own constraint,
  which is what let the loop decide what to retry.
* Failures are data too. Collect them, write them somewhere the person who typed them will look, and
  fix only what can be fixed without guessing.

### Cleanup

In [ ]:
invoke_sql_query(connection=sql_connection, query="DROP TABLE dbo.ProjectStatus", enable_exception=True)

if os.path.exists(failures_path):
    os.remove(failures_path)

sql_connection.close()